In [0]:
import os
import random
from datetime import datetime, timedelta
import json
from faker import Faker
import pandas as pd

In [0]:
fake = Faker()
Faker.seed(42)
random.seed(42)

In [0]:
spark.sql('create catalog if not exists DE;')
spark.sql('create schema if not exists DE.P1;')
spark.sql('create volume if not exists DE.P1.raw_volume;')

DataFrame[]

In [0]:
BASE_DIR = '/Volumes/de/p1/raw_volume'
os.makedirs(os.path.join(BASE_DIR, 'users'), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, 'products'), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, 'orders'), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, 'events'), exist_ok=True)

print("Mock e-commerce data is generating ...")

Mock e-commerce data is generating ...


In [0]:
users_data = []
for user_id in range(1, 101):
    users_data.append({
        'user_id': user_id,
        'signup_date': (datetime.now() - timedelta(days=random.randint(10, 180))).strftime('%Y-%m-%d'),
        'tier': random.choice(['Bronze', 'Silver', 'Gold', 'Platinum']),
        'city': fake.city(),
        'updated_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    })
df_users = pd.DataFrame(users_data)
df_users.to_parquet(os.path.join(BASE_DIR, 'users', 'users.parquet'), index=False)

In [0]:
categories = ['Electronics', 'Apparel', 'Home & Living', 'Beauty', 'Books']
products_data = []
for prod_id in range(1, 51):
    products_data.append({
        'product_id': prod_id,
        'product_name': fake.catch_phrase(),
        'category': random.choice(categories),
        'price': round(random.uniform(10.0, 1500.0), 2),
        'updated_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    })
df_products = pd.DataFrame(products_data)
df_products.to_parquet(os.path.join(BASE_DIR, 'products', 'products.parquet'), index=False)

In [0]:
start_date = datetime.now() - timedelta(days=3)
for day_offset in range(3):
    current_date = start_date + timedelta(days=day_offset)
    date_str = current_date.strftime('%Y-%m-%d')

    orders_data = []
    for order_id in range(day_offset * 200 + 1, (day_offset + 1) * 200 + 1):
        orders_data.append({
            'order_id': order_id,
            'user_id': random.randint(1, 100),
            'product_id': random.randint(1, 50),
            'quantity': random.randint(1, 5),
            'order_status': random.choice(['COMPLETED', 'PENDING', 'CANCELLED', 'REFUNDED']),
            'order_timestamp': (current_date + timedelta(hours = random.randint(0, 23), minutes = random.randint(0, 59))). strftime('%Y-%m-%d %H:%M:%S')
        })
    df_orders = pd.DataFrame(orders_data)
    day_dir = os.path.join(BASE_DIR, 'orders', f'{date_str}')
    os.makedirs(day_dir, exist_ok=True)
    df_orders.to_parquet(os.path.join(day_dir, 'orders.parquet'), index=False)

In [0]:
evnets_data = []
for _ in range(500):
    evnets_data.append({
        'event_id': fake.uuid4(),
        'user_id': random.choice([str(random.randint(1, 100)), None]),
        'event_type': random.choice(['page_view', 'click', 'add_to_cart', 'checkout']),
        'properties': json.dumps({'page': fake.uri_page(), 'device':random.choice(['iOS', 'Android', 'Web'])}),
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    })
    df_events = pd.DataFrame(evnets_data)
    df_events.to_json(os.path.join(BASE_DIR, 'events', 'clickstream.json'), orient='records', lines=True)

print('The mock data generating has been finished, and saved to the path: /Workspace/Users/winson_2397@hotmail.com/DE Project/P1/Data')

The mock data generating has been finished, and saved to the path: /Workspace/Users/winson_2397@hotmail.com/DE Project/P1/Data
